# Detección de Cumplimiento de EPP (Casco) en Obras de Construcción — YOLOv8

Notebook estilo **Google Colab**. Entrena y evalúa un detector YOLOv8 para verificar si los
trabajadores **usan o no casco**, con énfasis en **corta y larga distancia** (objetos pequeños),
robustez ante **oclusión** y un **verificador de cumplimiento basado en reglas**.

Dataset: `large-benchmark-datasets/logistics-sz9jr` v2 (Roboflow Universe, CC BY 4.0).


## 1. Instalación de dependencias


In [ ]:
!pip install -q ultralytics roboflow opencv-python


## 2. Comprobar GPU
En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4).


In [ ]:
!nvidia-smi


## 3. Obtener el código del proyecto
Clona el repositorio (sustituye la URL por la tuya).


In [ ]:
!git clone https://github.com/USUARIO/ppe-helmet-yolov8.git
%cd ppe-helmet-yolov8


## 4. Configurar la API key de Roboflow
No la dejes escrita en el repositorio. En Colab usa `getpass`.


In [ ]:
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')


### (Referencia) Snippet de descarga original de Roboflow
El proyecto encapsula esto en `src/download_data.py`.


In [ ]:
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
# project = rf.workspace('large-benchmark-datasets').project('logistics-sz9jr')
# version = project.version(2)
# dataset = version.download('yolov8')


## 5. Descargar el dataset (~95k imágenes, 20 clases)


In [ ]:
!python src/download_data.py


## 6. Construir subconjunto enfocado en EPP
El dataset completo (66k imágenes de entrenamiento) es inviable para 100 épocas en una sola GPU.
Creamos un subconjunto de 4 clases (`person, helmet, safety vest, gloves`) priorizando imágenes
con casco. Se respeta la división original train/valid/test.


In [ ]:
!python src/prepare_subset.py --train 5000 --valid 1200 --test 2000


## 7. Análisis exploratorio (desbalance y tamaños/distancia)


In [ ]:
!python src/analyze_dataset.py --data data/ppe_subset/data.yaml
from IPython.display import Image, display
display(Image('outputs/dataset_class_distribution.png'))
display(Image('outputs/dataset_bbox_size_distribution.png'))


## 8. Entrenamiento (YOLOv8s · imgsz=640 · 100 épocas)
Decisiones: backbone preentrenado en COCO; arquitectura **anchor-free**; **mosaic** activado con
`close_mosaic=10`; `scale=0.5` para simular variaciones de distancia. `batch=8` por la VRAM disponible.


In [ ]:
!python src/train.py --data data/ppe_subset/data.yaml --model yolov8s.pt \
    --epochs 100 --imgsz 640 --batch 8 --device 0 --name ppe_yolov8s


In [ ]:
display(Image('runs/detect/ppe_yolov8s/results.png'))


## 9. Evaluación: mAP por clase + robustez (distancia y oclusión)


In [ ]:
!python src/evaluate.py --data data/ppe_subset/data.yaml \
    --weights runs/detect/ppe_yolov8s/weights/best.pt --split test --device 0
display(Image('outputs/robustness_by_distance.png'))
display(Image('outputs/robustness_by_occlusion.png'))


## 10. Verificador de cumplimiento + demo sobre imágenes no vistas
Asocia espacialmente `person`↔`helmet`: una persona sin casco asociado se marca como **VIOLACIÓN**.
Genera fotogramas anotados, un clip de video y la tasa de cumplimiento por fotograma.


In [ ]:
!python src/infer_demo.py --weights runs/detect/ppe_yolov8s/weights/best.pt \
    --source data/ppe_subset/test/images --max 80 --make-video --device 0
import glob
for p in sorted(glob.glob('outputs/demo/frames/*.jpg'))[:4]:
    display(Image(p))


## (Alternativa) Pipeline completo con un solo comando
Todo lo anterior está orquestado en `run_pipeline.py`:


In [ ]:
# !python run_pipeline.py


## Conclusiones
- El modelo detecta personas y cascos; el verificador convierte detecciones en una tasa de
  cumplimiento y marca violaciones.
- La detección de **cascos lejanos (objetos pequeños)** es el régimen más difícil: ver la caída de
  recall en el grupo *small* del análisis por distancia.
- Implicaciones éticas (falsos negativos vs. positivos, privacidad, sesgo): ver `report/REPORT.md`.
